In [1]:
import os
from pathlib import Path
# 创建项目目录结构
# 获取当前脚本所在目录的绝对路径
current_dir = Path.cwd()  # 或者 Path(".").absolute()
project_root = current_dir / "ship_detection"
dataset_dir = project_root / "datasets" / "seaships"
models_dir = project_root / "models"
results_dir = project_root / "results"

print(f"项目根目录: {project_root}")

项目根目录: D:\ShipTarget\ship_detection


In [2]:
import json
import cv2

def yolo_to_coco(images_dir, labels_dir, classes_path, info=None):
    # 读取类别
    with open(classes_path, 'r') as f:
        classes = [line.strip() for line in f.readlines() if line.strip()]

    coco = {
        "info": info or {"description": "SeaShips dataset"},
        "images": [],
        "annotations": [],
        "categories": [{"id": i+1, "name": name} for i, name in enumerate(classes)]  # id 从 1 开始
    }

    ann_id = 1
    for img_path in sorted(images_dir.glob("*.*")):
        # 读取图像尺寸
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"警告：无法读取图像 {img_path}，跳过")
            continue
        img_height, img_width = img.shape[:2]

        # 图像信息
        image_info = {
            "id": len(coco["images"]) + 1,
            "file_name": img_path.name,
            "width": img_width,
            "height": img_height
        }
        coco["images"].append(image_info)

        # 对应的标签文件（假设标签文件与图像同文件名，扩展名为.txt）
        label_path = labels_dir / f"{img_path.stem}.txt"
        if not label_path.exists():
            continue

        with open(label_path, 'r') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            class_id, x_center, y_center, width, height = map(float, parts)
            # 转换为绝对坐标
            x = (x_center - width/2) * img_width
            y = (y_center - height/2) * img_height
            w = width * img_width
            h = height * img_height
            # 确保坐标不超出图像范围（可选）
            x = max(0, min(x, img_width - 1))
            y = max(0, min(y, img_height - 1))
            w = min(w, img_width - x)
            h = min(h, img_height - y)

            annotation = {
                "id": ann_id,
                "image_id": image_info["id"],
                "category_id": int(class_id) + 1,  # 如果类别 id 从 0 开始，需加 1
                "bbox": [x, y, w, h],
                "area": w * h,
                "iscrowd": 0
            }
            coco["annotations"].append(annotation)
            ann_id += 1

    return coco
# YOLO格式转COCO格式

# 定义需要处理的子集
splits = ['train', 'val', 'test']
for split in splits:
    print(f"正在处理 {split} 集...")
    # 对应的图像和标签子目录
    images_split_dir = dataset_dir / "images" / split
    labels_split_dir = dataset_dir / "labels" / split

    # 调用 yolo_to_coco 转换
    coco_data = yolo_to_coco(
        images_dir=images_split_dir,
        labels_dir=labels_split_dir,
        classes_path=dataset_dir / "classes.txt",
        info={"description": f"SeaShips {split} set for Faster R-CNN"}
    )

    # 保存为对应的 JSON 文件
    coco_annotation_dir = dataset_dir / "annotations"
    coco_annotation_dir.mkdir(exist_ok=True)
    output_json = coco_annotation_dir / f"instances_{split}.json"

    with open(output_json, "w") as f:
        json.dump(coco_data, f, indent=2)
    print(f"{split} 集转换完成，共 {len(coco_data['images'])} 张图像，标注文件保存至 {output_json}")
    
print("COCO格式标注文件已创建")

正在处理 train 集...
train 集转换完成，共 3559 张图像，标注文件保存至 D:\ShipTarget\ship_detection\datasets\seaships\annotations\instances_train.json
正在处理 val 集...
val 集转换完成，共 508 张图像，标注文件保存至 D:\ShipTarget\ship_detection\datasets\seaships\annotations\instances_val.json
正在处理 test 集...
test 集转换完成，共 1018 张图像，标注文件保存至 D:\ShipTarget\ship_detection\datasets\seaships\annotations\instances_test.json
COCO格式标注文件已创建


In [24]:
# 创建Faster R-CNN配置文件
faster_rcnn_config = """
# 模型配置
model = dict(
    type='FasterRCNN',
    data_preprocessor=dict(
        type='DetDataPreprocessor',
        mean=[123.675, 116.28, 103.53],   
        std=[58.395, 57.12, 57.375],      
        bgr_to_rgb=True,                  
        pad_size_divisor=32
    ),             
    backbone=dict(
        type='ResNet',
        depth=50,
        num_stages=4,
        out_indices=(0, 1, 2, 3),
        frozen_stages=1,
        norm_cfg=dict(type='BN', requires_grad=True),
        norm_eval=True,
        style='pytorch',
        init_cfg=dict(type='Pretrained', checkpoint='torchvision://resnet50')
    ),
    neck=dict(
        type='FPN',
        in_channels=[256, 512, 1024, 2048],
        out_channels=256,
        num_outs=5
    ),
    rpn_head=dict(
        type='RPNHead',
        in_channels=256,
        feat_channels=256,
        anchor_generator=dict(
            type='AnchorGenerator',
            scales=[8],
            ratios=[0.5, 1.0, 2.0],
            strides=[4, 8, 16, 32, 64]
        ),
        bbox_coder=dict(
            type='DeltaXYWHBBoxCoder',
            target_means=[.0, .0, .0, .0],
            target_stds=[1.0, 1.0, 1.0, 1.0]
        ),
        loss_cls=dict(
            type='CrossEntropyLoss', use_sigmoid=True, loss_weight=1.0),
        loss_bbox=dict(type='L1Loss', loss_weight=1.0)
    ),
    roi_head=dict(
        type='StandardRoIHead',
        bbox_roi_extractor=dict(
            type='SingleRoIExtractor',
            roi_layer=dict(type='RoIAlign', output_size=7, sampling_ratio=0),
            out_channels=256,
            featmap_strides=[4, 8, 16, 32]
        ),
        bbox_head=dict(
            type='Shared2FCBBoxHead',
            in_channels=256,
            fc_out_channels=1024,
            roi_feat_size=7,
            num_classes=1,  # 根据您的数据集类别数修改
            bbox_coder=dict(
                type='DeltaXYWHBBoxCoder',
                target_means=[0., 0., 0., 0.],
                target_stds=[0.1, 0.1, 0.2, 0.2]
            ),
            reg_class_agnostic=False,
            loss_cls=dict(
                type='CrossEntropyLoss', use_sigmoid=False, loss_weight=1.0),
            loss_bbox=dict(type='L1Loss', loss_weight=1.0)
        )
    ),
    train_cfg=dict(
        rpn=dict(
            assigner=dict(
                type='MaxIoUAssigner',
                pos_iou_thr=0.7,
                neg_iou_thr=0.3,
                min_pos_iou=0.3,
                match_low_quality=True,
                ignore_iof_thr=-1),
            sampler=dict(
                type='RandomSampler',
                num=256,
                pos_fraction=0.5,
                neg_pos_ub=-1,
                add_gt_as_proposals=False),
            allowed_border=-1,
            pos_weight=-1,
            debug=False),
        rpn_proposal=dict(
            nms_pre=2000,
            max_per_img=1000,
            nms=dict(type='nms', iou_threshold=0.7),
            min_bbox_size=0),
        rcnn=dict(
            assigner=dict(
                type='MaxIoUAssigner',
                pos_iou_thr=0.5,
                neg_iou_thr=0.5,
                min_pos_iou=0.5,
                match_low_quality=False,
                ignore_iof_thr=-1),
            sampler=dict(
                type='RandomSampler',
                num=512,
                pos_fraction=0.25,
                neg_pos_ub=-1,
                add_gt_as_proposals=True),
            pos_weight=-1,
            debug=False)
    ),
    test_cfg=dict(
        rpn=dict(
            nms_pre=1000,
            max_per_img=1000,
            nms=dict(type='nms', iou_threshold=0.7),
            min_bbox_size=0),
        rcnn=dict(
            score_thr=0.05,
            nms=dict(type='nms', iou_threshold=0.5),
            max_per_img=100)
    )
)

# 数据集配置
dataset_type = 'CocoDataset'
data_root = 'D:/ShipTarget/ship_detection/datasets/seaships/'

metainfo = dict(classes=['ship'])

train_dataloader = dict(
    batch_size=2,
    num_workers=0,
    persistent_workers=False,
    drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=True),
    batch_sampler=dict(type='AspectRatioBatchSampler'),
    collate_fn=dict(type='pseudo_collate'),
    dataset=dict(
        type=dataset_type,
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/instances_train.json',
        data_prefix=dict(img='images/train/'),
        filter_cfg=dict(filter_empty_gt=True, min_size=32),
        pipeline=[
            dict(type='LoadImageFromFile'),
            dict(type='LoadAnnotations', with_bbox=True),
            dict(type='Resize', scale=(640, 640), keep_ratio=True),
            dict(type='RandomFlip', prob=0.5),
            dict(type='PackDetInputs')
        ])
)

val_dataloader = dict(
    batch_size=8,
    num_workers=0,
    persistent_workers=False,
    drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False),
    collate_fn=dict(type='pseudo_collate'),
    dataset=dict(
        type=dataset_type,
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/instances_val.json',  
        data_prefix=dict(img='images/val/'),
        test_mode=True,
        pipeline=[
            dict(type='LoadImageFromFile'),
            dict(type='LoadAnnotations', with_bbox=True),
            dict(type='Resize', scale=(640, 640), keep_ratio=True),
            dict(type='PackDetInputs')
        ])
)

test_dataloader = dict(
    batch_size=8,
    num_workers=0,
    persistent_workers=False,
    drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False),
    collate_fn=dict(type='pseudo_collate'),
    dataset=dict(
        type=dataset_type,
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/instances_test.json',
        data_prefix=dict(img='images/test/'),
        test_mode=True,
        pipeline=[
            dict(type='LoadImageFromFile'),
            dict(type='LoadAnnotations', with_bbox=True),
            dict(type='Resize', scale=(640, 640), keep_ratio=True),
            dict(type='PackDetInputs')
        ])
)

# 训练配置
train_cfg = dict(type='EpochBasedTrainLoop', max_epochs=50, val_interval=5)
val_cfg = dict(type='ValLoop')
test_cfg = dict(type='TestLoop')

# 优化器配置
optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=dict(type='SGD', lr=0.02, momentum=0.9, weight_decay=0.0001)
)

# 学习率调度
param_scheduler = [
    dict(type='LinearLR', start_factor=0.001, by_epoch=False, begin=0, end=500),
    dict(type='MultiStepLR', milestones=[30, 40], gamma=0.1)
]

# 评估指标
val_evaluator = dict(
    type='CocoMetric',
    ann_file=data_root + 'annotations/instances_val.json',
    metric=['bbox'],
    format_only=False)

test_evaluator = dict(
    type='CocoMetric',
    ann_file=data_root + 'annotations/instances_test.json',
    metric=['bbox'],
    format_only=False)
"""

# 保存配置文件
config_path = models_dir / "faster_rcnn_config.py"
with open(config_path, "w", encoding='utf-8') as f:
    f.write(faster_rcnn_config)

print(f"Faster R-CNN配置文件已保存: {config_path}")

Faster R-CNN配置文件已保存: D:\ShipTarget\ship_detection\models\faster_rcnn_config.py


In [20]:
import sys
import torch
from mmdet.utils import register_all_modules
# 注册所有模块
register_all_modules()

import mmcv
from mmengine.config import Config
from mmengine.runner import Runner

# 加载配置
cfg = Config.fromfile(str(config_path))
cfg.val_dataloader.batch_size = 2
cfg.test_dataloader.batch_size = 2

# 设置工作目录
cfg.work_dir = str(results_dir / "faster_rcnn" / "baseline")

# 设置设备
cfg.device = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg.default_scope = 'mmdet'

# 创建runner并开始训练
runner = Runner.from_cfg(cfg)
runner.train()

print("Faster R-CNN训练完成")

03/11 23:17:48 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: win32
    Python: 3.9.25 (main, Nov  3 2025, 22:44:01) [MSC v.1929 64 bit (AMD64)]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 92328886
    GPU 0: NVIDIA GeForce RTX 3050 Laptop GPU
    CUDA_HOME: None
    MSVC: n/a, reason: fileno
    PyTorch: 2.1.0+cu118
    PyTorch compiling details: PyTorch built with:
  - C++ Version: 199711
  - MSVC 192930151
  - Intel(R) Math Kernel Library Version 2020.0.2 Product Build 20200624 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 2019
  - LAPACK is enabled (usually provided by MKL)
  - CPU capability usage: AVX512
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=compute_37,code=sm_37;-gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;arch=compute_61,code=sm_

In [31]:
# 加载训练好的模型进行测试
cfg = Config.fromfile(str(config_path))
cfg.work_dir = str(results_dir / "faster_rcnn" / "baseline")
cfg.device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg.default_scope = 'mmdet'
cfg.load_from = str(results_dir / "faster_rcnn" / "baseline" / "epoch_50.pth")

# 创建测试runner
runner = Runner.from_cfg(cfg)
metrics = runner.test()

print("Faster R-CNN评估结果:")
print(metrics)

03/12 09:56:03 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: win32
    Python: 3.9.25 (main, Nov  3 2025, 22:44:01) [MSC v.1929 64 bit (AMD64)]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 695303590
    GPU 0: NVIDIA GeForce RTX 3050 Laptop GPU
    CUDA_HOME: None
    MSVC: n/a, reason: fileno
    PyTorch: 2.1.0+cu118
    PyTorch compiling details: PyTorch built with:
  - C++ Version: 199711
  - MSVC 192930151
  - Intel(R) Math Kernel Library Version 2020.0.2 Product Build 20200624 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 2019
  - LAPACK is enabled (usually provided by MKL)
  - CPU capability usage: AVX512
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=compute_37,code=sm_37;-gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;arch=compute_61,code=sm